# MàthemaSharp.LinearAlgebra — Usage

LU con pivoting parziale e QR per piccoli sistemi reali. Prerequisiti: build Release, JupyterNet su .NET 11. La libreria LinearAlgebra è net10.0, ma i test e questo notebook usano anche i formati .NET 11.

## Contratti
- LU: matrice quadrata, non vuota, finita; rifiuta pivot numericamente piccoli. La fattorizzazione è riutilizzabile e indipendente dalla matrice originale. Solve richiede un termine noto finito e compatibile.
- Determinant può superare il range del tipo; un risultato piccolo non basta a diagnosticare il condizionamento. Inverse risolve per le colonne dell'identità; per Ax=b preferire Solve.
- LeastSquares richiede righe ≥ colonne, dati finiti e colonne indipendenti; normalizza il sistema, usa Householder e non forma XᵀX. Intermedi non rappresentabili sono segnalati.
- NormL2 e FrobeniusNorm usano Hypot. Multiply/Identity ammettono INumber; LU, norme e QR richiedono IEEE.


In [ ]:
#load "../shared/MathemaSharp.Numerics.csx"
using System.Numerics;
using System.Globalization;
using System.Runtime.InteropServices;
using MathemaSharp.LinearAlgebra;
Console.WriteLine(RuntimeInformation.FrameworkDescription);


.NET 11.0.0-rc.3.26522.148

## LU: pivoting, residuo e riuso
Il primo pivot è zero: senza scambio delle righe il procedimento non può partire. La soluzione attesa è [1,2,3]. Il residuo è verificato sui dati originali, non soltanto sulla soluzione attesa.


In [ ]:
double[,] a = {{0,2,1},{1,1,0},{2,0,1}};
double[] b = [7,3,5];
var lu = LuFactorization<double>.Create(a);
var x = lu.Solve(b);
var reconstructed = MatrixAlgebra.Multiply(a,x);
var residual = reconstructed.Zip(b,(left,right)=>left-right).ToArray();
if (MatrixAlgebra.NormL2<double>(residual) > 1e-12) throw new Exception("LU residual");
if (Math.Abs(lu.Determinant()+4)>1e-12) throw new Exception("Determinant");
var identity = MatrixAlgebra.Multiply(a,lu.Inverse());
for(var i=0;i<3;i++) for(var j=0;j<3;j++) if(Math.Abs(identity[i,j]-(i==j?1:0))>1e-12) throw new Exception("Inverse");
var again = lu.Solve(b);
if (!again.SequenceEqual(x)) throw new Exception("Reuse");
Console.WriteLine("lu_solution: "+string.Join(", ",x));
Console.WriteLine("lu_residual: OK");
Console.WriteLine("inverse_and_reuse: OK");


lu_solution: 1, 2, 3
lu_residual: OK
inverse_and_reuse: OK

## QR in Decimal128, senza conversioni da double
I dati sono creati da interi. Le tolleranze sono espresse in T. Un caso esatto piccolo non certifica tutte le matrici o tutti i formati.


In [ ]:
var one = Decimal128.One;
var zero = Decimal128.Zero;
var two = Decimal128.CreateChecked(2);
var three = Decimal128.CreateChecked(3);
var five = Decimal128.CreateChecked(5);
Decimal128[,] design = {{one,zero},{zero,one},{one,one}};
var beta = MatrixAlgebra.LeastSquares<Decimal128>(design,[two,three,five]);
var eps = Decimal128.BitIncrement(one)-one;
if(Decimal128.Abs(beta[0]-two)>Decimal128.CreateChecked(64)*eps || Decimal128.Abs(beta[1]-three)>Decimal128.CreateChecked(64)*eps) throw new Exception("QR Decimal128");
Console.WriteLine("qr_decimal128: OK");
Console.WriteLine("norm_l2: "+MatrixAlgebra.NormL2<double>([3,4]).ToString(CultureInfo.InvariantCulture));


qr_decimal128: OK
norm_l2: 5

## Reference e limiti
[Catalogo API pubblico](12-api-reference.ipynb) · [Verifiche sui sei formati](11-consolidation.ipynb) · [Inventario](../../ARCHITECTURE.md).

Questa libreria non è BLAS, non stima automaticamente il condizionamento e non offre ancora SVD generale. Le soglie di rango/pivot possono rifiutare matrici mal condizionate. La normalizzazione può fallire se y/scala non è rappresentabile. Per misure usare BenchmarkDotNet, non il processo del notebook.
